# Latency, Cost Budgets, and Caching
Companion notebook for the **Latency, Cost Budgets, and Caching** section of Chapter 8.

Even a safe agent can be slow and expensive if nothing bounds it. Set budgets, and degrade in a controlled way when a run goes past them.

The book's budget dict and check, with a small `run`/`fallback_response` stand-in so it actually executes:

In [1]:
class Run:
    def __init__(self, latency_ms, cost_usd):
        self.latency_ms = latency_ms
        self.cost_usd = cost_usd


def fallback_response(message: str) -> dict:
    return {"status": "degraded", "message": message}


budget = {
    "max_ms": 8000,
    "max_tokens": 12000,
    "max_tool_calls": 4,
    "max_cost_usd": 0.08,
}


def check_budget(run):
    if run.latency_ms > budget["max_ms"] or run.cost_usd > budget["max_cost_usd"]:
        return fallback_response("Budget exceeded")
    return {"status": "ok"}

In [2]:
print(check_budget(Run(latency_ms=3200, cost_usd=0.03)))
print(check_budget(Run(latency_ms=9100, cost_usd=0.03)))
print(check_budget(Run(latency_ms=3200, cost_usd=0.15)))

{'status': 'ok'}
{'status': 'degraded', 'message': 'Budget exceeded'}
{'status': 'degraded', 'message': 'Budget exceeded'}


**Exact-match caching** reuses a result for a stable prompt/task instead of paying the same latency and token cost again. The cache key includes model version, prompt version, and tenant -- not just the raw text -- so results never leak across contexts that shouldn't share them:

In [3]:
import time

CACHE_TTL_S = 300
_cache: dict[tuple, tuple[float, str]] = {}
call_count = {"expensive_call": 0}


def expensive_call(prompt: str) -> str:
    call_count["expensive_call"] += 1
    return f"answer for: {prompt}"


def cached_call(prompt: str, model_version: str, prompt_version: str, tenant: str) -> str:
    key = (model_version, prompt_version, tenant, prompt)
    cached = _cache.get(key)
    if cached and (time.monotonic() - cached[0]) < CACHE_TTL_S:
        return cached[1]
    result = expensive_call(prompt)
    _cache[key] = (time.monotonic(), result)
    return result


cached_call("What is our refund policy?", "gpt-4o", "v3", "tenant-acme")
cached_call("What is our refund policy?", "gpt-4o", "v3", "tenant-acme")  # same key -> cache hit
cached_call("What is our refund policy?", "gpt-4o", "v3", "tenant-globex")  # different tenant -> miss

print("expensive_call invocations:", call_count["expensive_call"], "(expected 2: one miss per distinct tenant, one hit)")

expensive_call invocations: 2 (expected 2: one miss per distinct tenant, one hit)


And the simplest saving of all: not every step needs an LLM. Routing, validation, and formatting can often be plain code, as the earlier notebooks in this chapter already show (`route_request`, `safe_read_file`, `call_with_retry` are all rule-based, zero-token).